In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import numpy as np
import xarray as xr
import cfgrib
import logging

logging.getLogger("cfgrib").setLevel(logging.WARNING)


def _parse_and_normalize(yaml_req: Optional[dict], dims_seen: set[str]) -> Dict[str, dict]:
    """
    Normalise le YAML en:
      group_name -> {
        "dimension": str | None,
        "variables": list[str],
        "all_levels": bool,
        "levels": list[float|int],
        "level_indices": list[int],
      }
    - 'dimension' pris du YAML si fourni, sinon détecté si le nom de groupe est une coordonnée vue.
    - Si aucune dimension, le groupe est traité comme surface-like.
    """
    if not yaml_req:
        return {}
    out: Dict[str, dict] = {}
    for group, spec in yaml_req.items():
        spec = spec or {}
        dim = spec.get("dimension", None)
        if dim is None and group in dims_seen:
            dim = group
        sel = str(spec.get("level_selection", "")).lower().strip()
        all_levels = sel == "all"
        if sel not in {"", "values", "indices", "all"}:
            sel = "values" if spec.get("level_values") else "indices"
        levels = list(spec.get("level_values", [])) if sel == "values" else []
        idxs   = list(spec.get("level_indices", [])) if sel == "indices" else []
        out[group] = {
            "dimension": dim,
            "variables": list(spec.get("variables", [])),
            "all_levels": bool(all_levels),
            "levels": levels,
            "level_indices": idxs,
        }
    return out


def _match_levels(avail_vals, requested, tol: float) -> List[float]:
    if not requested:
        return []
    vals = np.asarray(avail_vals, dtype=float)
    out = []
    for lv in requested:
        lvf = float(lv)
        j = int(np.argmin(np.abs(vals - lvf)))
        if abs(float(vals[j]) - lvf) <= tol:
            out.append(float(vals[j]))
    seen = set(); uniq = []
    for v in out:
        if v not in seen:
            seen.add(v); uniq.append(v)
    return uniq


def _collect_parts_for_req(
    dsets: List[xr.Dataset],
    req_map: Dict[str, dict],
    float_tol: float,
    tag: str,  # "user variables dict" | "tracker variables dict"
) -> Dict[str, List[xr.Dataset]]:
    """
    Parcourt les blocs et prépare les morceaux Ã  fusionner pour un req_map donné.
    Retour: {group: [datasets filtrés pour ce fichier]}
    """
    file_parts: Dict[str, List[xr.Dataset]] = {}

    for ds in dsets:
        for g, req in req_map.items():
            dim = req.get("dimension")
            if dim and dim not in ds.dims:
                continue
            sel = ds
            if "valid_time" in sel.coords:
                sel = sel.assign_coords(time=sel["valid_time"])

            # Sélection verticale
            if dim and dim in sel.dims and not req.get("all_levels", False):
                if req.get("level_indices"):
                    idx_list = [int(i) for i in req["level_indices"]]
                    n = int(sel.dims[dim])
                    valid_idx = [i for i in idx_list if -n <= i < n]
                    if not valid_idx:
                        continue
                    sel = sel.isel({dim: valid_idx})
                elif req.get("levels"):
                    matched = _match_levels(sel[dim].values, req["levels"], tol=float_tol)
                    if not matched:
                        continue
                    sel = sel.sel({dim: matched})

            # Variables
            keep = [v for v in req.get("variables", []) if v in sel.data_vars]
            if keep:
                file_parts.setdefault(g, []).append(sel[keep])

    return file_parts


def concat_grib2ds_by_vert_coord(
    files,
    user_requested_variables_yaml: dict,
    tracker_requested_variables_by_method_yaml: dict,
    method: Optional[str] = None,
    index_dir: str | Path = ".cfgrib",
    float_tol: float = 0.51,
    warn: bool = True,
    log: Optional[logging.Logger] = None,
) -> Tuple[Dict[str, xr.Dataset], Dict[str, Dict[str, xr.Dataset]]]:
    """
    Retourne:
      - by_group_user: {groupe -> xr.Dataset}
      - by_group_trk:  {méthode -> {groupe -> xr.Dataset}}
    """
    log = log or logging.getLogger("frameit")

    files = [Path(f) for f in files]
    idx_dir = Path(index_dir); idx_dir.mkdir(parents=True, exist_ok=True)

    # 1) Ouverture et inventaire des coordonnées
    opened_per_file: Dict[Path, List[xr.Dataset]] = {}
    dims_seen: set[str] = set()
    for f in files:
        dsets = cfgrib.open_datasets(str(f), indexpath=str(idx_dir / (f.name + ".idx")))
        opened_per_file[f] = dsets
        for ds in dsets:
            dims_seen.update(ds.dims.keys())

    # 2) Normalisation YAML
    user_req = _parse_and_normalize(user_requested_variables_yaml, dims_seen)

    tracker_yaml = tracker_requested_variables_by_method_yaml or {}
    if method is None:
        methods = list(tracker_yaml.keys())
    else:
        methods = [m for m in tracker_yaml.keys() if m == method]
        if not methods and warn:
            log.warning("Méthode tracker '%s' absente, aucun bloc tracker produit.", method)
    tracker_req_by_method: Dict[str, Dict[str, dict]] = {
        m: _parse_and_normalize(tracker_yaml.get(m, {}), dims_seen) for m in methods
    }

    # 3) Union pour inventaire global
    all_groups: Dict[str, dict] = dict(user_req)
    for m, blk in tracker_req_by_method.items():
        for k, v in blk.items():
            if k not in all_groups:
                all_groups[k] = v

    global_levels: Dict[str, set] = {g: set() for g, s in all_groups.items() if s.get("dimension")}
    global_depth:  Dict[str, int] = {g: 0 for g, s in all_groups.items() if s.get("dimension")}

    for _, dsets in opened_per_file.items():
        for ds in dsets:
            for g, spec in all_groups.items():
                dim = spec.get("dimension")
                if dim and dim in ds.dims:
                    global_depth[g] = max(global_depth[g], int(ds.dims[dim]))
                    try:
                        vals = np.asarray(ds[dim].values, dtype=float).ravel().tolist()
                    except Exception:
                        vals = [float(v) for v in ds[dim].values]
                    global_levels[g].update(vals)

    # 4) Sélection par fichier
    by_group_user: Dict[str, xr.Dataset] = {}
    by_group_trk: Dict[str, Dict[str, xr.Dataset]] = {m: {} for m in methods}

    for f, dsets in opened_per_file.items():
        parts_user = _collect_parts_for_req(dsets, user_req, float_tol, tag="user variables dict")
        parts_trk_by_m: Dict[str, Dict[str, List[xr.Dataset]]] = {
            m: _collect_parts_for_req(dsets, tracker_req_by_method[m], float_tol, tag=f"tracker variables dict:{m}")
            for m in methods
        }

        # Fusion intra-fichier puis concat
        for g, parts in parts_user.items():
            ds_file = xr.merge(parts, compat="override", join="outer", combine_attrs="drop_conflicts")
            by_group_user[g] = (
                xr.concat([by_group_user[g], ds_file], dim="time").sortby("time")
                if g in by_group_user else ds_file
            )

        for m, parts_trk in parts_trk_by_m.items():
            for g, parts in parts_trk.items():
                ds_file = xr.merge(parts, compat="override", join="outer", combine_attrs="drop_conflicts")
                if g in by_group_trk[m]:
                    by_group_trk[m][g] = xr.concat([by_group_trk[m][g], ds_file], dim="time").sortby("time")
                else:
                    by_group_trk[m][g] = ds_file

    return by_group_user, by_group_trk


In [ ]:
files = sorted(Path("/cnrm/mlac/NO_SAVE/souffletc/DONEE_TEST_FRAMEIT/TEST_GRIB_FILES/short_grib_file/")
               .glob("arome_BATSIRAI_20220201+??00P.grib"))

# YAML utilisateur
user_yaml = {
    "level": {"variables": ["u","v"], "level_selection": "indices", "level_indices": [0, 5, 10]},
    "isobaricInhPa": {"variables": ["u","v"], "level_selection": "values", "level_values": [1000, 850, 400]},
    "heightAboveGround": {"variables": ["u","v","t","r","tke"], "level_selection": "indices", "level_indices": [0,2,5]},
    "surface": {"variables": ["u10","v10","prmsl","sshf","slhf"]},
}

# YAML tracker
trk_yaml_by_method = {
    "wind_pressure": {
        "surface": {"variables": ["u10", "v10", "prmsl"]}
    },
    "other_method": {
        "isobaricInhPa": {"variables": ["u","v","z","pv"], "level_selection": "indices", "level_indices": [0,5,7]},
        "surface": {"variables": ["prmsl"]},
        "heightAboveGround": {"variables": ["u","t"], "level_selection": "indices", "level_indices": [0,2,5]}
    }
}
        
by_user, by_tracker = concat_grib2ds_by_vert_coord(files, user_yaml, trk_yaml_by_method, method=None, index_dir=".cfgrib")

In [ ]:
by_user['heightAboveGround'].tke.isel(time=2,heightAboveGround=0).plot(xlim=(59,62))

In [ ]:
by_tracker.keys()

In [ ]:
ds_user.lev

In [ ]:
from pathlib import Path
import xarray as xr
import cfgrib

def open_grib_multi_dict(
    files,
    requests: dict,       
    index_dir: str | Path = ".cfgrib",
) -> dict[str, xr.Dataset]:
    """
    Ouvre des fichiers GRIB avec cfgrib, extrait plusieurs groupes verticaux et retourne
    un dictionnaire {groupe_vertical: Dataset} sans renommer les variables.

    Chaque groupe vertical est détecté à partir des dimensions présentes dans les blocs
    que cfgrib expose via `open_datasets`:
      - "isobaricInhPa" si la dimension `isobaricInhPa` existe
      - "heightAboveGround" si la dimension `heightAboveGround` existe
      - "surface" sinon

    Le paramètre `requests` pilote, par groupe, la sélection des variables et des niveaux.

    Paramètres
    ----------
    files : iterable de str ou Path
        Liste ordonnée des fichiers GRIB à lire.
    requests : dict
        Spécifications par groupe vertical. Pour chaque clé parmi
        {"isobaricInhPa", "heightAboveGround", "surface"}, fournir un dict optionnel:
          - "variables": list[str]
                Noms de variables à extraire. Seules celles présentes sont gardées.
          - "levels": list[int | float]
                Niveaux à garder par valeur. Ignoré si "level_indices" est fourni.
          - "level_indices": list[int]
                Indices de niveaux à garder, relatifs à l'ordre des niveaux dans chaque bloc.
                Les indices hors plage sont signalés par un message et ignorés.
          - "all_levels": bool
                Si True, conserve tous les niveaux pour ce groupe et ignore "levels" et "level_indices".
        Exemple:
            {
              "isobaricInhPa":     {"variables": ["u","v","t"], "levels": [200, 850]},
              "heightAboveGround": {"variables": ["t","r"], "level_indices": [0, 3]},
              "surface":           {"variables": ["sp","prmsl"]}
            }
    index_dir : str | Path, par défaut ".cfgrib"
        Dossier où cfgrib écrit ses fichiers d'index (.idx). Améliore la vitesse et la stabilité.

    Retour
    ------
    dict[str, xr.Dataset]
        Un dictionnaire mappant le nom du groupe vertical vers un Dataset concaténé sur l'axe `time`.
        Les variables gardent leurs noms d'origine. Les niveaux sont filtrés selon `requests`.

    Comportement et messages
    ------------------------
    - Les blocs d'un même fichier et d'un même groupe sont fusionnés puis ajoutés comme une seule
      pièce dans la concaténation temporelle.
    - Si des indices de niveaux sont hors plage pour un fichier et un groupe donnés, un message
      informatif est imprimé et ces indices sont ignorés.
    - Si des niveaux par valeur sont demandés mais absents de l'ensemble des blocs de ce fichier
      et de ce groupe, un message indique les niveaux manquants avec un aperÃ§u des niveaux disponibles.
    - Aucun message ne provoque d'exception. La fonction n'échoue pas à cause d'un niveau manquant.

    Exemples
    --------
    >>> files = sorted(Path("data").glob("arome_BATSIRAI_20220201+??00P.grib"))
    >>> requests = {
    ...   "isobaricInhPa":     {"variables": ["u","v","t"], "levels": [200, 850]},
    ...   "heightAboveGround": {"variables": ["t","r"], "level_indices": [0, 3]},
    ...   "surface":           {"variables": ["sp","prmsl"]},
    ... }
    >>> by_group = open_grib_multi_dict(files, requests, index_dir=".cfgrib")
    >>> list(by_group.keys())
    ['isobaricInhPa', 'heightAboveGround', 'surface']
    """
    files = [Path(f) for f in files]
    idx_dir = Path(index_dir); idx_dir.mkdir(parents=True, exist_ok=True)

    by_group: dict[str, xr.Dataset] = {}

    for f in files:
        dsets = cfgrib.open_datasets(str(f), indexpath=str(idx_dir / (f.name + ".idx")))

        # Regroupe les blocs par groupe vertical détecté
        grouped: dict[str, list[xr.Dataset]] = {}
        for ds in dsets:
            is_plev = "isobaricInhPa" in ds.dims
            is_zagl = "heightAboveGround" in ds.dims
            group = "isobaricInhPa" if is_plev else ("heightAboveGround" if is_zagl else "surface")
            if group in requests:
                grouped.setdefault(group, []).append(ds)

        file_parts_by_group: dict[str, list[xr.Dataset]] = {}

        for group, blocks in grouped.items():
            req = requests[group]
            all_levels    = bool(req.get("all_levels", False))
            level_indices = req.get("level_indices")
            levels        = req.get("levels")
            vars_req      = req.get("variables", [])

            dim = "isobaricInhPa" if group == "isobaricInhPa" else ("heightAboveGround" if group == "heightAboveGround" else None)

            # Vérifications globales pour limiter les faux positifs
            if dim and not all_levels:
                if level_indices is not None:
                    # vérifie la plage d'indices par rapport à la plus grande profondeur verticale des blocs
                    n_max = max((b.dims.get(dim, 0) for b in blocks if dim in b.dims), default=0)
                    idx_list = [int(i) for i in level_indices]
                    bad_idx = [i for i in idx_list if not (-n_max <= i < n_max)]
                    if bad_idx:
                        print(f"[open_grib_multi_dict] {f.name} group={group}, variable={vars_req}: out-of-range indices {bad_idx} for size {n_max}")
                elif levels:
                    # union des niveaux disponibles pour informer sur les niveaux absents
                    union_vals = set()
                    for b in blocks:
                        if dim in b.dims:
                            union_vals.update(b[dim].values.tolist())
                    missing = [lv for lv in levels if lv not in union_vals]
                    if missing:
                        avail_preview = sorted(list(union_vals))[:8]
                        print(f"[open_grib_multi_dict] {f.name} group={group}: missing levels {missing} (available examples: {avail_preview})")

            # Construction des morceaux par bloc
            for ds in blocks:
                if "valid_time" in ds.coords:
                    ds = ds.assign_coords(time=ds["valid_time"])

                # Sélection verticale locale
                if dim and not all_levels and dim in ds.dims:
                    if level_indices is not None:
                        n = ds.dims[dim]
                        idx_list = [int(i) for i in level_indices]
                        valid_idx = [i for i in idx_list if -n <= i < n]
                        if valid_idx:
                            ds = ds.isel({dim: valid_idx})
                    elif levels:
                        avail = ds[dim].values.tolist()
                        present = [lv for lv in levels if lv in avail]
                        if present:
                            ds = ds.sel({dim: present})

                # Filtre variables présentes
                keep_vars = [v for v in vars_req if v in ds.data_vars]
                if keep_vars:
                    file_parts_by_group.setdefault(group, []).append(ds[keep_vars])

        # Fusion intra-fichier par groupe, puis accumulation sur l'axe temps
        for g, parts in file_parts_by_group.items():
            ds_file = xr.merge(parts, compat="override", join="outer", combine_attrs="drop_conflicts")
            by_group[g] = xr.concat([by_group[g], ds_file], dim="time").sortby("time") if g in by_group else ds_file

    return by_group


In [ ]:
files = sorted(Path("/cnrm/mlac/NO_SAVE/souffletc/DONEE_TEST_FRAMEIT/TEST_GRIB_FILES/short_grib_file/")
               .glob("arome_BATSIRAI_20220201+??00P.grib"))
requests = {
    "isobaricInhPa":     {"variables": ["u","v","t"], "level_indices": [-1, 10]},  
    "heightAboveGround": {"variables": ["t","r"], "level_indices": [100, 10],"all_levels": True},    
    "surface":           {"variables": ["sp","prmsl"]},
}
by_group = open_grib_multi_dict(files, requests, index_dir=".cfgrib")

In [ ]:
by_group["heightAboveGround"]

In [ ]:
from pathlib import Path
import cfgrib, xarray as xr
import numpy as np

def _detect_group(ds: xr.Dataset) -> str:
    if "isobaricInhPa" in ds.dims: return "isobaricInhPa"
    if "heightAboveGround" in ds.dims: return "heightAboveGround"
    return "surface"

def _vdim(group: str) -> str | None:
    return {"isobaricInhPa": "isobaricInhPa", "heightAboveGround": "heightAboveGround"}.get(group)

def _as_floats_flat(vals) -> list[float]:
    flat = []
    if vals is None:
        return flat
    def _walk(x):
        if isinstance(x, (list, tuple, set, np.ndarray)):
            for y in x: _walk(y)
        else:
            # attempt numeric conversion
            flat.append(float(x))
    _walk(vals)
    return flat

def _match_levels(avail_vals, requested_vals, atol=1e-6):
    avail = np.asarray(_as_floats_flat(avail_vals), dtype=float)
    present, missing = [], []
    for lv in requested_vals or []:
        lvf = float(lv)
        m = np.isclose(avail, lvf, rtol=0.0, atol=atol)
        if m.any(): present.append(float(avail[np.argmax(m)]))
        else:       missing.append(lvf)
    # unique and sorted
    return sorted(set(present)), sorted(set(missing))

def plan_expected_extraction(files, requests: dict, index_dir: str | Path = ".cfgrib") -> dict:
    files = [Path(f) for f in files]
    idx_dir = Path(index_dir); idx_dir.mkdir(parents=True, exist_ok=True)

    union_vars   = {g: set() for g in requests}
    union_levels = {g: set() for g in requests}
    ref_levels   = {g: None for g in requests}  # reference level order for index selection

    for f in files:
        dsets = cfgrib.open_datasets(str(f), indexpath=str(idx_dir / (f.name + ".idx")))
        for ds in dsets:
            g = _detect_group(ds)
            if g not in requests: 
                continue
            req_vars = set(requests[g].get("variables", []))
            ds_vars  = set(ds.data_vars)
            if not (req_vars & ds_vars):
                continue  # only count blocks that have at least one requested var
            union_vars[g].update(req_vars & ds_vars)

            vdim = _vdim(g)
            if vdim and vdim in ds.dims:
                lv = ds[vdim].values.tolist()
                union_levels[g].update(_as_floats_flat(lv))
                if ref_levels[g] is None:
                    ref_levels[g] = _as_floats_flat(lv)  # keep the first seen ordering

    expected = {}
    for g, req in requests.items():
        vdim = _vdim(g)
        exp_vars = sorted(union_vars[g])
        exp_lv = None
        if vdim is not None:
            if req.get("all_levels", False):
                exp_lv = sorted(union_levels[g])
            elif req.get("levels") is not None:
                present, _missing = _match_levels(union_levels[g], req["levels"])
                exp_lv = present
            elif req.get("level_indices") is not None:
                base = ref_levels[g] or []
                n = len(base)
                exp_lv = [base[i] for i in req["level_indices"] if -n <= int(i) < n]
        expected[g] = {"variables": exp_vars, "levels": exp_lv, "vdim": vdim}
    return expected

def validate_extraction_report(by_group: dict[str, xr.Dataset], requests: dict, expected: dict, atol=1e-6) -> dict:
    report = {}
    for g, req in requests.items():
        item = {"group_present": g in by_group, "missing_vars": [], "missing_levels": [], "extra_levels": [], "ok": False}
        if g not in by_group:
            report[g] = item
            continue

        ds = by_group[g]
        exp_vars = set(expected[g]["variables"])
        got_vars = set(v for v in ds.data_vars if v in req.get("variables", []))
        item["missing_vars"] = sorted(exp_vars - got_vars)

        vdim = expected[g]["vdim"]
        exp_lv = expected[g]["levels"]
        if vdim and vdim in ds.dims and exp_lv is not None:
            got = np.asarray(_as_floats_flat(ds[vdim].values), dtype=float)
            missing = []
            for lv in exp_lv:
                if not np.isclose(got, float(lv), rtol=0.0, atol=atol).any():
                    missing.append(float(lv))
            item["missing_levels"] = sorted(missing)

        item["ok"] = (item["group_present"] and not item["missing_vars"] and not item["missing_levels"])
        report[g] = item
    return report

# -------------------------
# Exemple d'usage
# -------------------------
if __name__ == "__main__":
    files = sorted(Path("/cnrm/mlac/NO_SAVE/souffletc/DONEE_TEST_FRAMEIT/TEST_GRIB_FILES/short_grib_file/")
               .glob("arome_BATSIRAI_20220201+??00P.grib"))

    requests = {
        "isobaricInhPa":     {"variables": ["u","v","t"], "levels": [200, 850]},
        "heightAboveGround": {"variables": ["t","r"], "level_indices": [0,2,3]},
        "surface":           {"variables": ["sp","prmsl"]},
    }

    # 1) Calcul de ce qui est attendu en scannant les GRIB
    expected = plan_expected_extraction(files, requests, index_dir=".cfgrib")

    # 2) Extraction avec ta fonction existante
    by_group = open_grib_multi_dict(files, requests, index_dir=".cfgrib")

    # 3) Validation
    report = validate_extraction_report(by_group, requests, expected)

    # 4) Affichage lisible
    import pprint
    pprint.pprint(report)
    # Tu peux aussi décider d'échouer en mode strict, par exemple:
    # assert all(v["ok"] for v in report.values()), "Extraction mismatch, see report"

In [ ]:
by_group["heightAboveGround"].r.isel(time=2,heightAboveGround=2).plot()

In [ ]:
by_group["isobaricInhPa"].v.isel(time=4,isobaricInhPa=0).plot()

In [ ]:
by_group["surface"].prmsl.isel(time=4).plot()